# Preprocessing (`test_shop` + `set_a` + `set_c`): Unified Weekly Panel & Feature Frame

Combines **`test_shop` (`history.parquet`)** with **Dataset A (`set-a`)** and **Dataset C (`set-c`)** (excluding Dataset B and Dataset D), cleans and standardizes all three into a gap-free Monday weekly panel, engineers causal features and multi-horizon scaled targets (`target_h1..target_h13`), and exports `features_wide.parquet`.

- **Google Colab & local ready:** Searches `/content/`, `./data/`, and `.` for `history.parquet`, `set-a`, and `set-c` (or lets you point `DATA_DIRS` directly to your folder/Drive path).

In [ ]:
%pip install -q numpy pandas pyarrow

In [ ]:
import gc
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

IN_COLAB = Path('/content').exists()
OUT_DIR = Path('/content/processed') if IN_COLAB else Path('processed')

H = 13
SID = 'series_id'

# Sample weights: test_shop gets weight 1.0; auxiliary transfer sets A and C get down-weighted
# so global models prioritize target-shop dynamics while still learning shared cross-series patterns.
SOURCE_WEIGHTS = {
    'test_shop': 1.0,
    'set_a': 0.35,
    'set_c': 0.35,
}

## 1. Locate & Load `test_shop`, `set_a`, and `set_c`

Automatically discovers:
- **`test_shop`**: `history.parquet` (or `test-shop/history.parquet`)
- **`set_a`**: any parquet/csv file or subfolder matching `set-a` / `set_a` / `seta` (e.g. `set-a/sales.parquet` or `set_a.parquet`)
- **`set_c`**: any parquet/csv file or subfolder matching `set-c` / `set_c` / `setc` (e.g. `set-c/sales.parquet` or `set_c.parquet`)

In [ ]:
# Optional explicit overrides (leave None for automatic discovery)
EXPLICIT_PATHS = {
    'test_shop': None,
    'set_a': None,
    'set_c': None,
}

SEARCH_ROOTS = [Path('/content'), Path('.')] if IN_COLAB else [Path('.'), Path('data'), Path('data/hackathon-retail-sets')]


def parse_week_start(s):
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_datetime(s, unit='D', origin='1970-01-01')
    return pd.to_datetime(s)


def _read_table(path):
    p = Path(path)
    if p.suffix.lower() == '.parquet':
        return pd.read_parquet(p, engine='pyarrow')
    if p.suffix.lower() == '.csv':
        return pd.read_csv(p)
    raise ValueError(f'Unsupported file extension: {p}')


def discover_dataset_files(roots=SEARCH_ROOTS):
    all_files = []
    for r in roots:
        if r.exists():
            for ext in ('*.parquet', '*.csv'):
                all_files.extend([p for p in r.rglob(ext) if p.stat().st_size > 0 and 'processed' not in p.parts])

    found = {'test_shop': [], 'set_a': [], 'set_c': []}
    for p in sorted(set(all_files)):
        low = str(p).lower().replace('\\', '/')
        name = p.name.lower()
        if name.startswith('features_wide') or name.startswith('submission'):
            continue
        if 'history' in name or 'test-shop' in low or 'test_shop' in low:
            found['test_shop'].append(p)
        elif any(tok in low for tok in ('set-a', 'set_a', '/seta', 'seta.')):
            found['set_a'].append(p)
        elif any(tok in low for tok in ('set-c', 'set_c', '/setc', 'setc.')):
            found['set_c'].append(p)

    resolved = {}
    for key in ('test_shop', 'set_a', 'set_c'):
        if EXPLICIT_PATHS[key] is not None:
            p = Path(EXPLICIT_PATHS[key])
            resolved[key] = [p] if p.is_file() else sorted(p.rglob('*.parquet')) + sorted(p.rglob('*.csv'))
        else:
            resolved[key] = found[key]
    return resolved


FILES = discover_dataset_files()
for k, v in FILES.items():
    print(f'{k:<10s}: {[str(p) for p in v] if v else "NOT FOUND"}')

missing = [k for k, v in FILES.items() if not v]
if missing:
    raise FileNotFoundError(
        f'Missing files for {missing}. Upload `history.parquet`, `set-a`, and `set-c` or set EXPLICIT_PATHS above.'
    )

## 2. Standardize & Build Unified Weekly Panel (`test_shop` + `set_a` + `set_c`)

Each dataset is normalized to a shared schema:
- `source`: `'test_shop'`, `'set_a'`, or `'set_c'`
- `series_id`: `'<source>|<store_id>|<article_id>'` (or `'<source>|<article_id>'` when single-store)
- `week_start`: aligned to Monday (`W-MON`)
- `units`: non-negative float weekly unit sales, gap-filled with `0.0` across each series' active/full span
- **Memory-safe series filtering:** drops all-zero series from auxiliary sets `set_a` and `set_c` (while keeping **all** `test_shop` articles so the submission index is 100% preserved).

In [ ]:
DATE_CANDIDATES = ['week_start', 'week', 'date', 'ds', 'timestamp', 'period_start']
UNIT_CANDIDATES = ['units', 'qty', 'quantity', 'sales', 'demand', 'y', 'units_sold']
ITEM_CANDIDATES = ['article_id', 'item_id', 'sku', 'product_id', 'id', 'series_id']
STORE_CANDIDATES = ['store_id', 'shop_id', 'location_id', 'branch_id', 'store']
STATIC_CANDIDATES = [
    'supplier_id', 'product_group', 'division', 'department', 'department_group',
    'store_format', 'category', 'subcategory', 'segment'
]


def _pick_col(cols, candidates, required=True, label=''):
    low_map = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand in low_map:
            return low_map[cand]
    if required:
        raise KeyError(f'Could not find {label} column in {list(cols)} (tried {candidates})')
    return None


def load_and_standardize_source(source_name, paths, drop_all_zero_series=False):
    t0 = time.perf_counter()
    # If multiple files exist in a folder (e.g. sales.parquet + meta.parquet), pick the time-series table
    ts_tables = []
    meta_tables = []
    for p in paths:
        df = _read_table(p)
        cols_low = {c.lower() for c in df.columns}
        has_date = any(c in cols_low for c in DATE_CANDIDATES)
        has_unit = any(c in cols_low for c in UNIT_CANDIDATES)
        if has_date and has_unit:
            ts_tables.append(df)
        else:
            meta_tables.append(df)

    if not ts_tables:
        raise ValueError(f'No time-series file with date & units columns found for {source_name} in {paths}')

    raw = pd.concat(ts_tables, ignore_index=True)
    date_col = _pick_col(raw.columns, DATE_CANDIDATES, label='date')
    unit_col = _pick_col(raw.columns, UNIT_CANDIDATES, label='units')
    item_col = _pick_col(raw.columns, ITEM_CANDIDATES, label='article_id')
    store_col = _pick_col(raw.columns, STORE_CANDIDATES, required=False)

    df = pd.DataFrame({
        'article_id': raw[item_col].astype(str),
        'week_start': parse_week_start(raw[date_col]).dt.to_period('W-SUN').dt.start_time,
        'units': pd.to_numeric(raw[unit_col], errors='coerce').fillna(0.0).clip(lower=0.0).astype(np.float32),
    })
    if store_col is not None:
        df['store_id'] = raw[store_col].astype(str)
        df[SID] = source_name + '|' + df['store_id'] + '|' + df['article_id']
    else:
        df['store_id'] = 'default'
        df[SID] = source_name + '|' + df['article_id']

    for extra in ['on_hand', 'unit_price', 'discount_pct']:
        if extra in raw.columns:
            df[extra] = pd.to_numeric(raw[extra], errors='coerce').astype(np.float32)

    # Aggregate any duplicate rows on the same (series_id, week_start)
    agg_dict = {'article_id': 'first', 'store_id': 'first', 'units': 'sum'}
    for extra in ['on_hand', 'unit_price', 'discount_pct']:
        if extra in df.columns:
            agg_dict[extra] = 'mean'
    df = df.groupby([SID, 'week_start'], as_index=False).agg(agg_dict)

    if drop_all_zero_series:
        tot = df.groupby(SID)['units'].transform('sum')
        df = df[tot > 0].reset_index(drop=True)

    # Build complete Monday weekly grid per series
    if source_name == 'test_shop':
        # Full dataset span for every test_shop article
        w_min, w_max = df['week_start'].min(), df['week_start'].max()
        sids = sorted(df[SID].unique())
        all_weeks = pd.date_range(w_min, w_max, freq='W-MON')
        grid = pd.MultiIndex.from_product([sids, all_weeks], names=[SID, 'week_start']).to_frame(index=False)
    else:
        # Active span (first observed week -> last observed week) per auxiliary series
        bounds = df.groupby(SID, as_index=False)['week_start'].agg(w_min='min', w_max='max')
        grid_parts = [
            pd.DataFrame({SID: row[SID], 'week_start': pd.date_range(row.w_min, row.w_max, freq='W-MON')})
            for row in bounds.itertuples(index=False)
        ]
        grid = pd.concat(grid_parts, ignore_index=True)

    id_map = df.groupby(SID, as_index=False)[['article_id', 'store_id']].first()
    grid = grid.merge(id_map, on=SID, how='left')

    dyn_cols = [SID, 'week_start', 'units'] + [c for c in ['on_hand', 'unit_price', 'discount_pct'] if c in df.columns]
    panel = grid.merge(df[dyn_cols], on=[SID, 'week_start'], how='left')
    panel['units'] = panel['units'].fillna(0.0).clip(lower=0.0).astype(np.float32)
    panel['source'] = source_name

    # Censorship flag when on_hand is available
    if 'on_hand' in panel.columns:
        prev_oh = panel.groupby(SID, sort=False)['on_hand'].shift(1)
        panel['censored'] = (panel['units'] == 0) & (prev_oh.notna()) & (prev_oh <= 0)
    else:
        panel['censored'] = False

    panel = panel.sort_values([SID, 'week_start']).reset_index(drop=True)
    print(f'Loaded {source_name:<10s} in {time.perf_counter() - t0:.1f}s: '
          f'{len(panel):,} rows | {panel[SID].nunique():,} series | '
          f'{panel["week_start"].min().date()} -> {panel["week_start"].max().date()}')
    return panel


panels = [
    load_and_standardize_source('test_shop', FILES['test_shop'], drop_all_zero_series=False),
    load_and_standardize_source('set_a',     FILES['set_a'],     drop_all_zero_series=True),
    load_and_standardize_source('set_c',     FILES['set_c'],     drop_all_zero_series=True),
]

panel = pd.concat(panels, ignore_index=True)
del panels
gc.collect()

panel['source'] = pd.Categorical(panel['source'], categories=['test_shop', 'set_a', 'set_c'])
panel = panel.sort_values(['source', SID, 'week_start']).reset_index(drop=True)

### Validate Combined Panel Invariants

In [ ]:
def check_panel(panel):
    allowed = {'test_shop', 'set_a', 'set_c'}
    assert set(panel['source'].unique()).issubset(allowed), f'Unexpected sources: {panel["source"].unique()}'
    assert not panel.duplicated([SID, 'week_start']).any(), 'Duplicate (series_id, week_start) rows'
    diffs = panel.groupby(SID, sort=False)['week_start'].diff().dropna()
    assert (diffs == pd.Timedelta(days=7)).all(), 'Non-7-day spacing detected in panel'
    assert (panel['units'].dropna() >= 0).all(), 'Negative units detected'
    summary_tbl = panel.groupby('source', observed=True).agg(
        rows=('units', 'size'),
        series=(SID, 'nunique'),
        min_week=('week_start', 'min'),
        max_week=('week_start', 'max'),
        mean_units=('units', 'mean'),
        zero_rate=('units', lambda s: (s == 0).mean()),
    )
    display(summary_tbl)


check_panel(panel)

## 3. Causal Feature Engineering (`features_wide`)

Computes identical scale-invariant features across `test_shop`, `set_a`, and `set_c`:
- **Lags & rolling stats:** `lag_1`, `lag_2`, `lag_3`, `lag_4`, `lag_8`, `lag_13`, `lag_26`, `lag_52`, rolling means/medians/stds/maxes/nonzero rates (`4w`, `8w`, `13w`, `26w`, `52w`), and trend ratios (`ratio_mean_4_13`, `ratio_mean_13_52`).
- **Intermittency & lifecycle:** `age_weeks`, `weeks_since_sale`, expanding `adi`, `cv2`, and Syntetos–Boylan `demand_class` (`smooth`, `erratic`, `intermittent`, `lumpy`, `no_sales`).
- **Scale normalization & weights:** `scale = max(roll_mean_13, 1.0)`, `sample_weight` from `SOURCE_WEIGHTS` (`1.0` for `test_shop`, `0.35` for `set_a` and `set_c`), `seasonal_h1..seasonal_h13`, and `target_h1..target_h13`.

In [ ]:
def classify_demand(adi, cv2, total_sales):
    conds = [
        total_sales <= 0,
        (adi < 1.32) & (cv2 < 0.49),
        (adi < 1.32) & (cv2 >= 0.49),
        (adi >= 1.32) & (cv2 < 0.49),
        (adi >= 1.32) & (cv2 >= 0.49),
    ]
    choices = ['no_sales', 'smooth', 'erratic', 'intermittent', 'lumpy']
    return np.select(conds, choices, default='no_sales')


def build_features_wide(panel, horizon=H, source_weights=SOURCE_WEIGHTS):
    t0 = time.perf_counter()
    df = panel.sort_values([SID, 'week_start']).reset_index(drop=True).copy()
    g = df.groupby(SID, sort=False)

    # Lifecycle & recency (strictly causal up to week t)
    df['age_weeks'] = g.cumcount().astype(np.int16)
    is_sale = (df['units'] > 0)
    sale_week_idx = np.where(is_sale, df['age_weeks'], np.nan)
    last_sale_idx = pd.Series(sale_week_idx).groupby(df[SID], sort=False).ffill().to_numpy()
    df['weeks_since_sale'] = (df['age_weeks'].to_numpy(float) - last_sale_idx).astype(np.float32)

    # Exact lags
    for l in [1, 2, 3, 4, 8, 13, 26, 52]:
        df[f'lag_{l}'] = g['units'].shift(l).astype(np.float32)

    # Rolling window stats ending at week t
    for w in [4, 8, 13, 26, 52]:
        r = g['units'].rolling(w, min_periods=1)
        df[f'roll_mean_{w}'] = r.mean().reset_index(level=0, drop=True).astype(np.float32)
        df[f'roll_median_{w}'] = r.median().reset_index(level=0, drop=True).astype(np.float32)
        df[f'roll_std_{w}'] = r.std().fillna(0.0).reset_index(level=0, drop=True).astype(np.float32)
        df[f'roll_max_{w}'] = r.max().reset_index(level=0, drop=True).astype(np.float32)

    df['is_sale_num'] = is_sale.astype(np.float32)
    g_sale = df.groupby(SID, sort=False)['is_sale_num']
    for w in [4, 13, 26, 52]:
        df[f'nz_rate_{w}'] = g_sale.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True).astype(np.float32)
    df.drop(columns=['is_sale_num'], inplace=True)

    # Trend / momentum ratios
    df['ratio_mean_4_13'] = (df['roll_mean_4'] / (df['roll_mean_13'] + 1e-3)).astype(np.float32)
    df['ratio_mean_13_52'] = (df['roll_mean_13'] / (df['roll_mean_52'] + 1e-3)).astype(np.float32)

    # Causal Syntetos-Boylan demand classification (expanding up to week t)
    exp_n = (df['age_weeks'].to_numpy(float) + 1.0)
    exp_nz = g['units'].transform(lambda s: (s > 0).cumsum()).to_numpy(float)
    exp_sum = g['units'].cumsum().to_numpy(float)
    pos_units = df['units'].where(df['units'] > 0)
    pos_mean = pos_units.groupby(df[SID], sort=False).expanding().mean().reset_index(level=0, drop=True).to_numpy(float)
    pos_std = pos_units.groupby(df[SID], sort=False).expanding().std().fillna(0.0).reset_index(level=0, drop=True).to_numpy(float)

    with np.errstate(divide='ignore', invalid='ignore'):
        adi = np.where(exp_nz > 0, exp_n / exp_nz, np.inf)
        cv2 = np.where(pos_mean > 0, (pos_std / pos_mean) ** 2, 0.0)
    df['adi'] = np.where(np.isfinite(adi), adi, 999.0).astype(np.float32)
    df['cv2'] = cv2.astype(np.float32)
    df['demand_class'] = pd.Categorical(
        classify_demand(adi, cv2, exp_sum),
        categories=['smooth', 'erratic', 'intermittent', 'lumpy', 'no_sales']
    )

    # Calendar features
    woy = df['week_start'].dt.isocalendar().week.to_numpy(float)
    df['week_of_year'] = woy.astype(np.int16)
    df['woy_sin'] = np.sin(2 * np.pi * woy / 52.0).astype(np.float32)
    df['woy_cos'] = np.cos(2 * np.pi * woy / 52.0).astype(np.float32)

    # Series scale & source-specific sample weight
    df['scale'] = np.maximum(df['roll_mean_13'].to_numpy(float), 1.0).astype(np.float32)
    df['is_outlier'] = False
    df['sample_weight'] = df['source'].astype(str).map(source_weights).fillna(0.35).astype(np.float32)

    # Multi-horizon seasonal features (52 - h weeks prior) and scaled targets (t + h)
    for h in range(1, horizon + 1):
        df[f'seasonal_h{h}'] = (g['units'].shift(52 - h) / df['scale']).astype(np.float32)
        df[f'target_h{h}'] = (g['units'].shift(-h) / df['scale']).astype(np.float32)

    print(f'Built features_wide in {time.perf_counter() - t0:.1f}s: {len(df):,} rows x {df.shape[1]} cols')
    return df


features_wide = build_features_wide(panel, horizon=H)
del panel
gc.collect()

print('Row & weight breakdown by source:')
display(features_wide.groupby('source', observed=True).agg(
    rows=('units', 'size'),
    series=(SID, 'nunique'),
    sample_weight=('sample_weight', 'first'),
    valid_h1_targets=('target_h1', lambda s: s.notna().sum()),
))

## 4. Export `features_wide.parquet`

Writes `features_wide.parquet` to `/content/processed/features_wide.parquet` (plus `/content/features_wide.parquet` in Colab) so it can be loaded directly into the modeling notebook.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUT_DIR / 'features_wide.parquet'
features_wide.to_parquet(out_path, engine='pyarrow', index=False)
print(f'Saved {out_path} ({out_path.stat().st_size / 1e6:.1f} MB)')

if IN_COLAB:
    root_copy = Path('/content/features_wide.parquet')
    features_wide.to_parquet(root_copy, engine='pyarrow', index=False)
    print(f'Also saved copy at {root_copy} for direct Colab session access')